# House Price Prediction Using Machine Learning

## Problem Understanding

The objective of this project is to predict house prices using property characteristics such as:

- Area of the property
- Number of rooms
- Build year
- Location
- Street type
- Furnishing status
- Property type
- Pool availability

### Machine Learning Concepts Used

- Data Cleaning
- Exploratory Data Analysis (EDA)
- Feature Engineering
- Data Preprocessing
- Linear Regression
- Random Forest Regression
- Model Evaluation
- Model Deployment

### Dataset Source

Dataset obtained from Kaggle and modified for educational purposes.

(Add your Kaggle dataset link here)


## Objectives

1. Analyze the housing dataset.
2. Perform data cleaning.
3. Create useful engineered features.
4. Visualize important trends.
5. Train machine learning models.
6. Compare model performance.
7. Predict house prices accurately.
8. Save the model for deployment.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

import joblib


## Load Dataset

In [ ]:
df = pd.read_csv('dataset_2.csv')
df.head()


## Dataset Information

In [ ]:
print("Dataset Shape:", df.shape)

df.info()


In [ ]:
df.describe(include='all')


## Dataset Description

Features:

- Area_SqFt : Total property area
- Rooms : Number of rooms
- Build_Year : Construction year
- Location : Property location
- Street_Type : Type of street access
- Furnishing : Furnished status
- Property_Type : Apartment, Villa, Duplex, etc.
- Has_Pool : Pool availability

Target Variable:

- Price


## Missing Values and Duplicates

In [ ]:
print(df.isnull().sum())
print("\nDuplicate Rows:", df.duplicated().sum())


In [ ]:
missing = df.isnull().sum()

plt.figure(figsize=(8,4))
plt.bar(missing.index, missing.values)
plt.xticks(rotation=45)
plt.title("Missing Values")
plt.show()


## Feature Engineering

Create a new feature:

### Property_Age

Property_Age = Current Year - Build_Year

This makes the feature easier to interpret than Build_Year.


In [ ]:
CURRENT_YEAR = 2026

df['Property_Age'] = CURRENT_YEAR - df['Build_Year']

df[['Build_Year','Property_Age']].head()


## Outlier Detection Using IQR

In [ ]:
plt.figure(figsize=(6,4))
plt.boxplot(df['Price'])
plt.title('Price Before Outlier Removal')
plt.show()


In [ ]:
Q1 = df['Price'].quantile(0.25)
Q3 = df['Price'].quantile(0.75)

IQR = Q3 - Q1

lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR

df = df[(df['Price'] >= lower) & (df['Price'] <= upper)]

print("Dataset Shape After Outlier Removal:", df.shape)


In [ ]:
plt.figure(figsize=(6,4))
plt.boxplot(df['Price'])
plt.title('Price After Outlier Removal')
plt.show()


## Exploratory Data Analysis

In [ ]:
plt.figure(figsize=(6,4))
plt.hist(df['Price'], bins=30)
plt.title('Price Distribution')
plt.show()


In [ ]:
plt.figure(figsize=(6,4))
plt.hist(df['Area_SqFt'], bins=30)
plt.title('Area Distribution')
plt.show()


In [ ]:
plt.figure(figsize=(6,4))
plt.hist(df['Rooms'], bins=20)
plt.title('Rooms Distribution')
plt.show()


In [ ]:
plt.figure(figsize=(6,4))
plt.scatter(df['Area_SqFt'], df['Price'])
plt.xlabel('Area')
plt.ylabel('Price')
plt.title('Area vs Price')
plt.show()


In [ ]:
plt.figure(figsize=(6,4))
plt.scatter(df['Rooms'], df['Price'])
plt.title('Rooms vs Price')
plt.show()


In [ ]:
plt.figure(figsize=(6,4))
plt.scatter(df['Property_Age'], df['Price'])
plt.title('Property Age vs Price')
plt.show()


In [ ]:
df['Location'].value_counts().plot(kind='bar', figsize=(8,4))
plt.title('Location Distribution')
plt.show()


In [ ]:
df['Property_Type'].value_counts().plot(kind='bar', figsize=(8,4))
plt.title('Property Type Distribution')
plt.show()


## Encoding Categorical Features

In [ ]:
df = pd.get_dummies(
    df,
    columns=[
        'Location',
        'Street_Type',
        'Furnishing',
        'Property_Type',
        'Has_Pool'
    ],
    drop_first=True
)

df.head()


## Correlation Analysis

In [ ]:
corr = df.corr(numeric_only=True)

plt.figure(figsize=(10,8))

plt.imshow(corr, cmap='coolwarm')
plt.colorbar()

plt.xticks(range(len(corr.columns)), corr.columns, rotation=90)
plt.yticks(range(len(corr.columns)), corr.columns)

plt.title('Correlation Heatmap')
plt.show()


## Train Test Split

In [ ]:
X = df.drop('Price', axis=1)
y = df['Price']

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print(X_train.shape)
print(X_test.shape)


## Feature Scaling

In [ ]:
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)


## Linear Regression

In [ ]:
lr = LinearRegression()

lr.fit(X_train, y_train)

lr_pred = lr.predict(X_test)


## Random Forest Regression

In [ ]:
rf = RandomForestRegressor(
    random_state=42
)

rf.fit(X_train, y_train)

rf_pred = rf.predict(X_test)


## Model Evaluation

In [ ]:
def evaluate_model(name, y_true, pred):

    mae = mean_absolute_error(y_true, pred)
    mse = mean_squared_error(y_true, pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_true, pred)

    print(f"\n{name}")
    print("MAE :", mae)
    print("MSE :", mse)
    print("RMSE:", rmse)
    print("R² :", r2)

    return r2


In [ ]:
lr_r2 = evaluate_model(
    "Linear Regression",
    y_test,
    lr_pred
)

rf_r2 = evaluate_model(
    "Random Forest",
    y_test,
    rf_pred
)


## Model Comparison

In [ ]:
plt.figure(figsize=(6,4))

plt.bar(
    ['Linear Regression','Random Forest'],
    [lr_r2, rf_r2]
)

plt.ylabel('R² Score')
plt.title('Model Comparison')

plt.show()


## Best Fit Line (Random Forest)

In [ ]:
plt.figure(figsize=(7,6))

plt.scatter(y_test, rf_pred)

plt.plot(
    [y_test.min(), y_test.max()],
    [y_test.min(), y_test.max()],
    'r'
)

plt.xlabel('Actual Price')
plt.ylabel('Predicted Price')

plt.title('Actual vs Predicted')

plt.show()


## Feature Importance

In [ ]:
feature_importance = pd.DataFrame({

    'Feature': X.columns,
    'Importance': rf.feature_importances_

})

feature_importance = feature_importance.sort_values(
    'Importance',
    ascending=False
)

plt.figure(figsize=(10,6))

plt.barh(
    feature_importance['Feature'],
    feature_importance['Importance']
)

plt.title('Feature Importance')

plt.show()

feature_importance.head(10)


## Sample Predictions

In [ ]:
comparison = pd.DataFrame({
    'Actual Price': y_test.values[:10],
    'Predicted Price': rf_pred[:10]
})

comparison


## Save Model

In [ ]:
joblib.dump(rf, 'house_price_model.pkl')
joblib.dump(scaler, 'scaler.pkl')

print('Model Saved Successfully')


## Streamlit Deployment

Create app.py separately.

Install:

```bash
pip install streamlit
```

Run:

```bash
streamlit run app.py
```


# Conclusion

- Successfully analyzed the housing dataset.
- Performed preprocessing and outlier removal.
- Created Property_Age feature.
- Trained Linear Regression and Random Forest models.
- Compared model performance using R² score.
- Saved the best model for deployment.

# Future Scope

- Larger datasets
- More location-based features
- XGBoost implementation
- Deep Learning models
- Real-time web deployment
